# 09 - Collect every result for the report

Reads what notebooks 00-08 saved on Drive and builds the report's tables and figures. Nothing is trained or attacked, so a **CPU runtime is enough** and it takes a minute. Re-run it whenever a notebook finishes; missing results are skipped.

**Output** (in `report_assets/` on Drive):
- `tables.tex`: every table as LaTeX (booktabs), ready for the AAAI template;
- `results_summary.md`: the same tables plus the claims they support, in plain words;
- `fig_main.png` (our method vs TMD, with 95% intervals), `fig_masking.png` (adaptive vs transfer), and `fig_off_manifold.png` (from notebook 08).

**How the tables map to the course's scoring criteria (Lecture 0):**

| Criterion (10 pts each) | Evidence produced here |
|---|---|
| Novelty and difficulty | the method (learned projection encoder, adversarial rounds, the BERT-level variant of 07b); the effort table (attack runs, sentences attacked, GPU hours) |
| Technical approach: correctness, good results | reproduction table next to the paper's numbers (no lowered baseline); main results with noise margins; paired significance tests; ablations; seeds; transfer and worst-case analysis |
| Writing | clear, small tables and three figures, each with one message |

In [ ]:
import os, json, math, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/tmd_run_paperft/ag_news'
OUT = f'{WORK}/report_assets'
os.makedirs(OUT, exist_ok=True)
N, N_CLEAN = 1000, 7600            # attacked test sentences; full AG News test set (clean accuracy)
M = ['clean', 'pwws', 'textfooler', 'bertattack']
NAME = {'clean': 'Clean', 'pwws': 'PWWS', 'textfooler': 'TextFooler', 'bertattack': 'BERT-Attack'}
PAPER = {'vanilla': dict(zip(M, (94.39, 39.2, 27.9, 39.0))), 'TMD': dict(zip(M, (94.29, 70.0, 50.0, 55.2))),
         'best baseline': dict(zip(M, (94.42, 39.3, 35.5, 52.3)))}   # Table 1, BERT, AG News

def load(tag, n=N, split='test'):
    p = f"{WORK}/results_fork_{'' if split == 'test' else split + '_'}{tag}_n{n}.json"
    return json.load(open(p)) if os.path.exists(p) else None

def row(tag, n=N, split='test'):
    r = load(tag, n, split)
    return None if r is None else {m: round(r[m]['acc'], 2 if m == 'clean' else 1) for m in M if m in r}

def ci(p, n):
    return 1.96 * math.sqrt(max(p / 100 * (1 - p / 100), 1e-4) / n) * 100

def verdict(ours, paper, m):
    n = N_CLEAN if m == 'clean' else N
    tol = math.sqrt(ci(ours, n) ** 2 + ci(paper, n) ** 2)
    d = ours - paper
    return d, tol, ('BEATS' if d > tol else 'worse' if d < -tol else 'tie')

def mcnemar(a, b, attack, n=N):
    def robust(tag):
        p = f'{WORK}/examples_fork_{tag}_n{n}_{attack}.jsonl'
        if not os.path.exists(p):
            return None
        return {r['i']: r['status'] == 'F' for r in map(json.loads, open(p)) if r['status'] != 'D'}
    ra, rb = robust(a), robust(b)
    if ra is None or rb is None:
        return None
    common = sorted(set(ra) & set(rb))
    oa, ob = sum(ra[i] and not rb[i] for i in common), sum(rb[i] and not ra[i] for i in common)
    k = oa + ob
    p = min(1.0, 2 * sum(math.comb(k, j) for j in range(min(oa, ob) + 1)) / 2 ** k) if k else 1.0
    return dict(attack=NAME[attack], sentences=len(common), only_a=oa, only_b=ob, p=round(p, 4))

def first_tested(tags):
    for t in tags:
        r = load(t)
        if r and all(m in r for m in ('pwws', 'textfooler', 'bertattack')):
            return t
    return None

nb08 = json.load(open(f'{WORK}/nb08_results.json')) if os.path.exists(f'{WORK}/nb08_results.json') else {}
OURS = 'enc_adv4_m9'
SEED2 = 'enc_adv4_s7_m9'
B7B = first_tested(['advbert2_enc2_swa', 'advbert2_enc2', 'advbert2_enc1', 'advbert2_tmd', 'advbert2'])
print('notebook 07b test result:', B7B or 'not available yet')
tex, md = [], []

def fmt(v):
    return '' if v is None or (isinstance(v, float) and math.isnan(v)) else (f'{v:.2f}' if isinstance(v, float) and v > 90 else str(v))

def emit(df, caption, label):
    # Append one table to tables.tex and results_summary.md, and show it.
    display(df)
    cols = list(df.columns)
    esc = lambda s: str(s).replace('_', r'\_').replace('%', r'\%')
    lines = [r'\begin{table}[t]', r'\centering', r'\small', r'\begin{tabular}{l' + 'r' * (len(cols) - 1) + '}',
             r'\toprule', ' & '.join(esc(c) for c in cols) + r' \\', r'\midrule']
    lines += [' & '.join(esc(fmt(v)) for v in r) + r' \\' for r in df.itertuples(index=False)]
    lines += [r'\bottomrule', r'\end{tabular}', rf'\caption{{{esc(caption)}}}', rf'\label{{{label}}}', r'\end{table}', '']
    tex.append('\n'.join(lines))
    md.append(f'### {caption}\n\n| ' + ' | '.join(cols) + ' |\n|' + '---|' * len(cols) + '\n'
              + '\n'.join('| ' + ' | '.join(fmt(v) for v in r) + ' |' for r in df.itertuples(index=False)) + '\n')

## Table 1: reproduction (our runs next to the paper's Table 1)

In [ ]:
van, tmd, t6 = row('vanilla'), row('baseline'), row('tmd_t6')
rep = []
for m in M:
    r = {'metric': NAME[m], 'paper vanilla': PAPER['vanilla'][m], 'our vanilla': van.get(m),
         'paper TMD': PAPER['TMD'][m], 'our TMD': tmd.get(m)}
    if t6:
        r['our TMD, Table 6 GAN'] = t6.get(m)
    if m != 'clean':
        r['TMD gain, paper'] = round(PAPER['TMD'][m] - PAPER['vanilla'][m], 1)
        r['TMD gain, ours'] = round(tmd[m] - van[m], 1)
    rep.append(r)
emit(pd.DataFrame(rep), 'Reproduction on AG News with BERT, using the authors\' attack code (1000 test sentences; clean accuracy on the full test set)', 'tab:repro')

## Table 2: main results against the paper's TMD

In [ ]:
defs = [('paper: TMD', PAPER['TMD']), ('paper: best baseline', PAPER['best baseline']), ('ours: TMD reproduced', tmd),
        ('ours: encoder, round 3 (enc_adv3)', row('enc_adv3_m9')), ('ours: encoder, round 4 (enc_adv4)', row(OURS))]
s2 = row(SEED2)
if s2:
    defs.append(('ours: enc_adv4, second seed', s2))
    o = row(OURS)
    defs.append(('ours: enc_adv4, mean of 2 seeds', {m: round((o[m] + s2[m]) / 2, 1) for m in s2}))
if B7B:
    defs.append((f'ours: BERT + encoder both adversarially trained ({B7B})', row(B7B)))
main = pd.DataFrame([{'defense': d, **{NAME[m]: r.get(m) for m in M}} for d, r in defs if r])
emit(main, 'Accuracy under attack (%) on AG News with BERT under the paper\'s protocol', 'tab:main')

claims = []
best_name, best = ('enc_adv4', row(OURS)) if not B7B else max([('enc_adv4', row(OURS)), (B7B, row(B7B))],
                                                                   key=lambda x: x[1]['textfooler'] + x[1]['bertattack'])
for m in M:
    if m in best:
        d, tol, v = verdict(best[m], PAPER['TMD'][m], m)
        claims.append(f'- {NAME[m]}: {best_name} {best[m]} vs paper TMD {PAPER["TMD"][m]} ({d:+.1f}, 95% noise ±{tol:.1f}): **{v}**')
md.append('### Claims against the paper\'s TMD\n\n' + '\n'.join(claims) + '\n')
print('\n'.join(claims))

## Table 3: paired significance against our TMD (same 1000 sentences, exact McNemar test)
`only TMD` / `only ours`: sentences that survive the attack under one defense but not the other.

In [ ]:
sig = []
for tag, label in [('enc_adv3_m9', 'enc_adv3'), (OURS, 'enc_adv4'), (SEED2, 'enc_adv4, seed 2')] + ([(B7B, B7B)] if B7B else []):
    for a in ('textfooler', 'bertattack', 'pwws'):
        r = mcnemar('baseline', tag, a)
        if r:
            sig.append({'defense': label, 'attack': r['attack'], 'only TMD': r['only_a'], 'only ours': r['only_b'], 'p': r['p']})
emit(pd.DataFrame(sig), 'Exact McNemar tests against our TMD reproduction on the same 1000 test sentences', 'tab:sig')

## Table 4: is TMD's robustness real? Adaptive attack, transfer, worst case, and the randomness-aware attack

In [ ]:
if 'worst_case' in nb08:
    wc = pd.DataFrame(nb08['worst_case'])
    wc['attack'] = wc.attack.map(NAME)
    wc['defense'] = wc.defense.replace({'baseline': 'TMD (ours)', OURS: 'enc_adv4', SEED2: 'enc_adv4, seed 2'})
    emit(wc[['attack', 'defense', 'adaptive', 'transfer', 'worst_case']],
         'Adaptive accuracy, accuracy on texts that fooled vanilla BERT (transfer), and per-sentence worst case', 'tab:masking')
if 'eot' in nb08:
    eot = pd.DataFrame(nb08['eot'])
    eot['attack'] = eot.attack.map(NAME)
    emit(eot, 'TMD against an attacker that sees its average output over 5 projections (first 300 test sentences)', 'tab:eot')

## Table 5: ablations on validation sentences (same 200 for every row)

In [ ]:
ABL = [('vanilla', 'vanilla BERT', 'no', 'no'), ('baseline', 'TMD (ours)', 'no', 'yes, random search'),
       ('tmd_t6', 'TMD with the paper\'s Table 6 GAN', 'no', 'yes, random search'),
       ('nrb_strong', 'BERT trained on generic noise (07)', 'no', 'no'),
       ('nrb_strong_tmd', 'noise-trained BERT + TMD (07)', 'no', 'yes, random search'),
       ('enc_den_m9', 'encoder, no attack examples', 'no', 'yes, learned'),
       ('mlp_adv4', 'classifier trained on attack examples, no GAN', 'yes', 'no'),
       ('enc_adv3_m9', 'encoder, round 3', 'yes', 'yes, learned'), ('enc_adv4_m9', 'encoder, round 4', 'yes', 'yes, learned'),
       ('enc_adv5_m9', 'encoder, round 5', 'yes', 'yes, learned'),
       ('advbert2', 'adversarially trained BERT alone (07b)', 'yes', 'no'),
       ('advbert2_tmd', 'adversarially trained BERT + TMD (07b)', 'yes', 'yes, random search'),
       ('advbert2_enc2', 'adversarially trained BERT + encoder (07b)', 'yes', 'yes, learned'),
       ('advbert2_enc2_swa', 'the same, weight-averaged encoder (07b)', 'yes', 'yes, learned')]
rows = []
for tag, desc, attack_aware, manifold in ABL:
    r = row(tag, 200, 'val')
    if r:
        rows.append({'variant': desc, 'learns from attacks': attack_aware, 'projection': manifold, 'Clean': r['clean'],
                     'TextFooler': r.get('textfooler'), 'BERT-Attack': r.get('bertattack')})
emit(pd.DataFrame(rows), 'Ablations on 200 validation sentences (accuracy under attack, %)', 'tab:ablation')

## Table 6: projection quality and cost; effort

In [ ]:
if 'reconstruction' in nb08:
    emit(pd.DataFrame(nb08['reconstruction']), 'Reconstruction error and clean accuracy of each projection (AG News test set)', 'tab:rec')
if 'timing' in nb08:
    emit(pd.DataFrame(nb08['timing']), 'Inference time on 1000 test sentences (the paper reports TMD at 2.07x vanilla, Table 5)', 'tab:time')

runs, sents, minutes = 0, 0, 0.0
for p in glob.glob(f'{WORK}/results_fork_*.json'):
    for k, v in json.load(open(p)).items():
        if k in ('pwws', 'textfooler', 'bertattack') and isinstance(v, dict):
            runs, sents, minutes = runs + 1, sents + v.get('n', 0), minutes + v.get('minutes', 0)
tr_runs, tr_sents, tr_sec = 0, 0, 0.0
for p in glob.glob(f'{WORK}/advpairs_*.jsonl'):
    rs = [json.loads(l) for l in open(p)]
    tr_runs, tr_sents, tr_sec = tr_runs + 1, tr_sents + len(rs), tr_sec + sum(r.get('seconds', 0) for r in rs)
effort = pd.DataFrame([{'what': 'evaluation attack runs (validation + test)', 'runs': runs, 'sentences attacked': sents,
                        'GPU hours': round(minutes / 60, 1)},
                       {'what': 'attack runs on training/validation sentences (training material)', 'runs': tr_runs,
                        'sentences attacked': tr_sents, 'GPU hours': round(tr_sec / 3600, 1)}])
emit(effort, 'Experimental effort (attack time only; fine-tuning and GAN training excluded)', 'tab:effort')

## Figures

In [ ]:
attacks = ['pwws', 'textfooler', 'bertattack']
bars = [('paper TMD', PAPER['TMD']), ('our TMD', tmd), ('ours: enc_adv4', row(OURS))] + ([(f'ours: {B7B}', row(B7B))] if B7B else [])
x, w = np.arange(len(attacks)), 0.8 / len(bars)
plt.figure(figsize=(6, 3))
for k, (lbl, r) in enumerate(bars):
    vals = [r[a] for a in attacks]
    plt.bar(x + k * w, vals, w, yerr=[ci(v, N) for v in vals], capsize=3, label=lbl)
plt.xticks(x + w * (len(bars) - 1) / 2, [NAME[a] for a in attacks]); plt.ylabel('accuracy under attack (%)')
plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{OUT}/fig_main.png', dpi=200); plt.show()

if 'worst_case' in nb08:
    wc = pd.DataFrame(nb08['worst_case'])
    wc = wc[wc.defense.isin(['baseline', OURS])]
    fig, axes = plt.subplots(1, 3, figsize=(8, 2.8), sharey=True)
    for ax, a in zip(axes, attacks):
        d = wc[wc.attack == a].set_index('defense')
        for k, (dfn, lbl) in enumerate([('baseline', 'TMD'), (OURS, 'ours')]):
            if dfn in d.index:
                ax.bar([k - 0.2, k + 0.2], [d.loc[dfn, 'adaptive'], d.loc[dfn, 'transfer']], 0.4,
                       color=['tab:gray', 'tab:orange'])
        ax.set_xticks([0, 1]); ax.set_xticklabels(['TMD', 'ours']); ax.set_title(NAME[a])
    axes[0].set_ylabel('accuracy (%)')
    from matplotlib.patches import Patch
    axes[0].legend(handles=[Patch(color='tab:gray', label='adaptive attack'),
                            Patch(color='tab:orange', label='transfer from vanilla')], fontsize=7)
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_masking.png', dpi=200); plt.show()

if os.path.exists(f'{OUT}/fig_off_manifold.png'):
    from IPython.display import Image
    display(Image(f'{OUT}/fig_off_manifold.png', width=450))

open(f'{OUT}/tables.tex', 'w').write('\n'.join(tex))
open(f'{OUT}/results_summary.md', 'w').write('# Results summary\n\n' + '\n'.join(md))
print('written:', sorted(os.listdir(OUT)))